In [ ]:
!pip install vitaldb
!pip install "pandas==2.2.2"

In [ ]:
import requests
import pandas as pd
import vitaldb
import io
import numpy as np
from tqdm import tqdm
import os
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
import neurokit2 as nk
from scipy import signal
from scipy.spatial.distance import cosine
from scipy.signal import find_peaks
import warnings
warnings.filterwarnings('ignore')

In [ ]:
RAWDATAPATH = 'drive/MyDrive/2025_PPG_GLUC/Data/Raw Data/'
VITALDB_DATAPATH = 'drive/MyDrive/2025_PPG_GLUC/Data/Raw Data/VitalDB/50Hz/'
OUTPUT_PATH = 'drive/MyDrive/2025_PPG_GLUC/Data/Processed Data/BW_ppg_16min_windows/'

INPUT_PATH = 'drive/MyDrive/2025_PPG_GLUC/Data/Processed Data/BW_ppg_16min_windows/'
OUTPUT_PATH2 = 'drive/MyDrive/2025_PPG_GLUC/Data/Processed Data/BW_ppg_16min_filtered/'
OUTPUT_PATH3 = Path('drive/MyDrive/2025_PPG_GLUC/Data/Final Data/1s_segmentation_adaptivev3/')
OUTPUT_PATH3.mkdir(parents=True, exist_ok=True)

In [ ]:
FS = 100 # Setting the sampling frequency

In [ ]:
os.listdir(VITALDB_DATAPATH)

## Downloading Lab and Clinical Data

In [ ]:
# Download Lab Data
labs_url = "https://api.vitaldb.net/labs"
labs_response = requests.get(labs_url)

if labs_response.status_code != 200:
    print(f"Failed to load labs data: {labs_response.status_code}")
else:
    labs_data = pd.read_csv(io.StringIO(labs_response.text))
    print(f"Total labs measurements: {len(labs_data)}")
    labs_data.to_csv(RAWDATAPATH + 'lab_data.csv')

# Download Clinical Data
clinical_url = "https://api.vitaldb.net/cases"
clinical_response = requests.get(clinical_url)

if clinical_response.status_code != 200:
    print(f"Failed to load clinical data: {clinical_response.status_code}")
else:
    clinical_data = pd.read_csv(io.StringIO(clinical_response.text))
    print(f"Clinical data loaded: {len(clinical_data)} cases")
    clinical_data.to_csv(RAWDATAPATH + 'clinical_data.csv')

## Downloading VitalDB

In [ ]:
# Find Cases with Glucose Lab Data
lab = pd.read_csv(RAWDATAPATH + 'lab_data.csv')
lab_gluc = lab[lab['name'] == 'gluc']
lab_gluc_cases = lab_gluc['caseid'].unique().tolist()

# Obtain PLETH (PPG) Data: Original PPG is 500 Hz, need to downsample to 50 Hz (from paper)
tracklist = ['SNUADC/PLETH']
existlist = [int(i.split('_')[-1].split('.')[0]) for i in os.listdir(VITALDB_DATAPATH + 'Vital/')]
to_retrieve = list(set(lab_gluc_cases) - set(existlist))
for caseid in tqdm(to_retrieve):
    tmp = vitaldb.load_case(caseid, tracklist, interval=1/50)
    np.save(VITALDB_DATAPATH + 'Vital/' + f'vitaldb_PLETH_case_{caseid}.npy', tmp)

# Re-try for those size ~1 KB
tracklist = ['SNUADC/PLETH']
existlist = [int(i.split('_')[-1].split('.')[0]) for i in os.listdir(VITALDB_DATAPATH + 'Vital/')]
original_list = [caseid for caseid in existlist if os.path.getsize(
    VITALDB_DATAPATH + 'Vital/' + f'vitaldb_PLETH_case_{caseid}.npy')/1024 <= 1]
print(len(original_list))

for caseid in tqdm(original_list):
    tmp = vitaldb.load_case(caseid, tracklist, interval=1/50)
    np.save(VITALDB_DATAPATH + 'Vital/' + f'vitaldb_PLETH_case_{caseid}.npy', tmp)

# Check again
after_list = [caseid for caseid in existlist if os.path.getsize(
    VITALDB_DATAPATH + 'Vital/' + f'vitaldb_PLETH_case_{caseid}.npy')/1024 <= 1]
print(len(after_list))

## PPG Temporal Alignment
* Find Caseids with Glucose Reading
* Filter Glucose Reading by Case Timings
* Generate PPG alignment

In [ ]:
labs_data = pd.read_csv(RAWDATAPATH + 'lab_data.csv').drop(columns=['Unnamed: 0'])
clinical_data = pd.read_csv(RAWDATAPATH + 'clinical_data.csv').drop(columns=['Unnamed: 0'])

labs_data.shape, clinical_data.shape, clinical_data['caseid'].nunique()

## Find Caseids with Glucose Reading

In [ ]:
glucose_data = labs_data[labs_data['name'] == 'gluc'].copy()
print(glucose_data.shape, glucose_data['caseid'].nunique())

# Glucose reading outside case time
print(glucose_data[glucose_data['dt']<0].shape[0]/glucose_data.shape[0])

## Filter Glucose Reading by Case Timings
Filters applied:
* dt >= casestart
* dt <= caseend

In [ ]:
glucose_with_timing = pd.merge(glucose_data,
                               clinical_data[['caseid', 'casestart', 'caseend']],
                               on='caseid', how='left')
print(clinical_data.shape, glucose_with_timing.shape)
print(clinical_data[clinical_data['casestart'].isna()].shape,
      clinical_data[clinical_data['caseend'].isna()].shape,
      glucose_with_timing[glucose_with_timing['dt'].isna()].shape,
      glucose_with_timing['caseid'].nunique())

# Valid time
valid_timing = glucose_with_timing[
        (glucose_with_timing['dt'] >= glucose_with_timing['casestart']) &
        (glucose_with_timing['dt'] <= glucose_with_timing['caseend'])
    ]
print(valid_timing.shape, valid_timing['caseid'].nunique())

## Generate PPG Temporal Alignment
*   Load PPG data from .npy files by case ID
*   Extract 16-minute windows
*   Temporal Alignment using timestamps

**Temporal Alignment Calculated Using**:
* `time_offset = glucose_time - case_start` - Time of glucose measurement relative to case start (in seconds)
* `center_sample = int(time_offset * sampling_rate)` - Exact sample index where glucose was measured
* `half_window = window_samples // 2` - Half of 16-minute window = 48,000 samples (8 minutes)
* `start_idx = center_sample - half_window` - Start of window (8 minutes before glucose measurement)
* `end_idx = center_sample + half_window` - End of window (8 minutes after glucose measurement)
* `ppg_window = ppg_data[start_idx:end_idx]` - Extract 96,000 samples (16 minutes at 100 Hz)

**Which cases are removed / avoided?**:
* Any case that the glucose reading is not within the case start / case end is not used
* Any case that has the glucose reading too close to the case start / case end where it is not able to extract the full 16 minute window
* Any case that has missing PPG data
* Any case that has misisng case timing information

In [ ]:
OUTPUT_PATH = 'drive/MyDrive/2025_PPG_GLUC/Data/Processed Data/50Hz_16min_windows/BW_ppg_16min_windows/'

In [ ]:
# Sort caseid
valid_timing = valid_timing.sort_values(by='caseid')

# Parameters
sampling_rate = 50
window_duration = 16 * 60  # seconds
window_samples = window_duration * sampling_rate
caseid_holder = int(-1)

# Generate meta data for tracking purpose
ppg_data_meta = pd.DataFrame({'Caseid':[], 'Gluc':[], 'Info':[]})

for index, row in tqdm(valid_timing.iterrows()):
    caseid = row['caseid']
    glucose_time = row['dt']
    case_start = row['casestart']
    filename = f"case_{caseid}_time_{glucose_time}_cleaned.npy"

    # Skip if already exist
    if os.path.exists(OUTPUT_PATH + filename):
        continue

    # Load Data
    if caseid != caseid_holder:  # Reload
        ppg_file = f"{RAWDATAPATH}VitalDB/50Hz/Vital/vitaldb_PLETH_case_{caseid}.npy"
        try:
            ppg_data = np.load(ppg_file)
            caseid_holder = caseid
        except:
            print(f'Error in loading vitaldb_PLETH_case_{caseid}.npy')
            continue  # skip to next

    # Extract window
    time_offset = glucose_time - case_start # to calculate relative time to case start
    center_sample = int(time_offset * sampling_rate) # to find the sampling index for gluc reading
    half_window = window_samples // 2 # for the 8 minute window

    start_idx = center_sample - half_window # start of window
    end_idx = center_sample + half_window # end of window

    if start_idx < 0 or end_idx >= len(ppg_data): # if ppg data out of the boundary, returns none
        print(f'Error in extracting {caseid}: out of boundary')
        continue
    else: # Forward filling missing
        cleaned_ppg_data = pd.Series(ppg_data[start_idx:end_idx].flatten()).ffill().values
        np.save(OUTPUT_PATH + filename, cleaned_ppg_data)

    # Note down the missing values that being forward filled
    original_nan = sum([1 for i in ppg_data.flatten() if np.isnan(i)])
    filled_nan = sum([1 for i in cleaned_ppg_data.flatten() if np.isnan(i)])
    ppg_data_meta = pd.concat([ppg_data_meta,
        pd.DataFrame({'Caseid':[caseid], 'Gluc': [glucose_time],
                      'Info': [f'Original missing: {original_nan}, After filling missing: {filled_nan}']})],
        ignore_index=True)

ppg_data_meta.to_csv(OUTPUT_PATH + 'BW_ppg_16min_metadata.csv', index=False)
ppg_data_meta.shape